# Notebook 6: Basic Data Analysis and Machine Learning

## Overview

This notebook demonstrates **exploratory data analysis (EDA) and machine learning classification** on preprocessed FTIR spectral data. You'll visualize spectral patterns, perform statistical analysis, apply dimensionality reduction, and build classification models.

### What You'll Learn

1. How to load and filter preprocessed FTIR data
2. How to visualize spectral patterns and variability
3. How to perform statistical analysis (ANOVA, correlation)
4. How to apply dimensionality reduction (PCA, t-SNE, UMAP, PLS-DA, OPLS-DA)
5. How to perform clustering analysis (K-means, hierarchical)
6. How to train and evaluate machine learning classification models
7. How to interpret model predictions with SHAP

### Prerequisites

You should have already:
- ✓ Completed Notebooks 1-5 OR have preprocessed FTIR data ready
- ✓ Data in CSV format with a label column (polymer type)
- ✓ Preprocessed data (denoised, baseline corrected, normalized)

### Analysis Workflow

This notebook follows a systematic analysis workflow:

1. **Data Loading & Filtering**: Load preprocessed data and select samples
2. **Exploratory Visualization**: Visualize spectral patterns and variability
3. **Statistical Analysis**: Identify significant wavenumbers (ANOVA, correlation)
4. **Dimensionality Reduction**: Reduce high-dimensional data for visualization
5. **Clustering Analysis**: Discover natural groupings in the data
6. **Machine Learning**: Build and evaluate classification models
7. **Model Interpretation**: Understand what drives predictions (SHAP)

### Available Analysis Methods

The `FTIRdataanalysis` class provides:

**Visualization:**
- Mean spectra by polymer type
- Overlay plots for comparison
- Coefficient of variation (CV) plots
- Correlation heatmaps

**Statistical Analysis:**
- ANOVA (identify discriminative wavenumbers)
- Correlation analysis

**Dimensionality Reduction:**
- PCA (Principal Component Analysis)
- t-SNE (t-Distributed Stochastic Neighbor Embedding)
- UMAP (Uniform Manifold Approximation and Projection)
- PLS-DA (Partial Least Squares Discriminant Analysis)
- OPLS-DA (Orthogonal PLS-DA)

**Clustering:**
- K-means clustering
- Hierarchical clustering

**Machine Learning:**
- 35+ pre-configured classification models
- Automated model comparison
- Hyperparameter tuning
- SHAP explainability analysis

### Expected Output

By the end of this notebook, you'll have:
- Visual understanding of spectral patterns
- Statistical insights into discriminative features
- Low-dimensional representations for visualization
- Trained classification models with performance metrics
- Interpretation of model predictions

---

## Step 1: Load Preprocessed Data

First, we'll load the preprocessed FTIR data created in previous notebooks.

In [ ]:
# Import required modules
import polars as pl
import pandas as pd
from xpectrass import FTIRdataanalysis

print("="*80)
print("LOADING PREPROCESSED DATA")
print("="*80)

In [ ]:
# Load the preprocessed data from Notebook 5
# Using 1st derivative data for enhanced spectral resolution
df = pd.read_csv('processed_data/combined_norm_data_0.csv.xz', compression='xz')

print(f"\nLoaded data shape: {df.shape}")
print(f"\nAvailable studies:")
print(df['study'].unique())
print("\nFirst few rows:")
print(df.head())

In [ ]:
print("\n" + "="*80)
print("FILTERING DATA")
print("="*80)

# Remove unknown samples dataset (focusing on labeled polymers)
df_ = df[df['study'] != 'kedzierski_2019_u']

print(f"\nAfter removing unknown samples: {df_.shape}")
print(f"\nUnique polymer types:")
print(df_['type'].unique())
print(f"\nSample distribution by polymer type:")
print(df_['type'].value_counts())

---

## Step 2: Filter and Select Data

Now we'll filter the data to focus on specific studies and polymer types with sufficient samples for analysis.

In [ ]:
print("\n" + "="*80)
print("SELECTING POLYMER TYPES WITH SUFFICIENT SAMPLES")
print("="*80)

# Select the 8 most common polymer types (sufficient samples for ML)
selected_types = ['PP', 'HDPE', 'PS', 'PET', 'LDPE', 'PVC', 'PE', 'PEF']
df_sel = df_[df_['type'].isin(selected_types)]

print(f"\nSelected polymer types: {selected_types}")
print(f"Selected data shape: {df_sel.shape}")

# For this example, exclude one study to create a balanced dataset
df_norm = df_sel[df_sel['study'] != 'villegas_camacho_2024_c4']

print(f"\nFinal dataset shape: {df_norm.shape}")
print(f"Final sample distribution:")
print(df_norm['type'].value_counts())
print("="*80)

---

## Step 3: Initialize FTIRdataanalysis

In [ ]:
print("\n" + "="*80)
print("INITIALIZING FTIRdataanalysis")
print("="*80)

# Initialize the analysis class
# This class handles all visualization, statistical analysis, and machine learning
fda = FTIRdataanalysis(
    df=df_norm,
    dataset_name='Combined dataset',
    label_column="type",
    exclude_columns=['stydy', 'sample_id', 'environmental', 'resulation'],  # Exclude metadata
    random_state=42,  # For reproducibility
    n_jobs=-1,  # Use all CPU cores for parallel processing
)

print(f"\n✓ FTIRdataanalysis initialized successfully")
print(f"  Dataset: Combined dataset")
print(f"  Samples: {len(df_norm)}")
print(f"  Features: {df_norm.shape[1] - len(['type', 'study', 'sample_id', 'environmental', 'resolution'])}")
print(f"  Polymer types: {df_norm['type'].nunique()}")
print("="*80)

---

## Step 4: Exploratory Data Visualization

### 4.1 Mean Spectra by Polymer Type

Visualize the mean (average) spectrum for each polymer type. This shows the characteristic spectral signatures that distinguish different polymers.

In [ ]:
# Plot mean spectra for each polymer type in separate subplots
# This shows the characteristic spectral pattern for each polymer
fda.plot_mean_spectra(
    title="Mean Spectra by Type",
    figsize=(10, 8),
    save_plot=True,
    save_path='mean_spectra',
)

### 4.2 Overlay Mean Spectra

Plot all mean spectra overlaid on the same axes for direct comparison. This helps identify spectral regions where polymers differ most.

In [ ]:
# Overlay all mean spectra on one plot for direct comparison
# Look for regions where spectra differ most - these are discriminative features
fda.plot_overlay_spectra(
    title="Mean Spectra overlay",
    figsize=(16, 4),
    save_plot=True,
    save_path="mean_spectra",
)

### 4.3 Coefficient of Variation (CV) Plot

The CV plot shows spectral variability within each polymer type. High CV indicates regions with high within-group variability, which may be due to sample heterogeneity or measurement noise.

In [ ]:
# Plot coefficient of variation (CV) to assess spectral variability
# Low CV = consistent spectra within polymer type
# High CV = high variability (may indicate sample heterogeneity)
fda.plot_cv(
    title="Spectral Variability by Type",
    figsize=(16, 4),
    save_plot=True,
    save_path="mean_spectra",
)

### 4.4 Correlation Heatmap

Shows mean spectra for all polymer types as a heatmap. Brighter regions indicate higher absorbance, darker regions indicate lower absorbance.

In [ ]:
# Heatmap showing mean spectra for all polymer types
# Columns = wavenumbers, Rows = polymer types
# Color intensity = absorbance value
fda.plot_heatmap(
    figsize=(5, 4),
    save_plot=True,
    save_path="mean_spectra",
)

---

## Step 5: Statistical Analysis

### 5.1 ANOVA Analysis

Analysis of Variance (ANOVA) identifies wavenumbers where polymer types differ significantly. High F-values indicate discriminative features.

In [ ]:
# Perform ANOVA to identify discriminative wavenumbers
# High F-values = large between-group variance relative to within-group variance
# These wavenumbers are most useful for classification
fda.perform_anova(
    figsize=(16, 8),
    save_plot=True,
    save_path="mean_spectra",
)

### 5.2 Correlation Analysis

Examines correlations between wavenumbers. Highly correlated features provide redundant information.

In [ ]:
# Correlation analysis between wavenumbers
# Dark red/blue = highly correlated features (redundant information)
# White = uncorrelated features (independent information)
fda.plot_correlation(
    figsize=(16, 12),
    save_plot=True,
    save_path="mean_spectra",
)

---

## Step 6*: remove zero interpolated part and reinitialize FTIRdataanalysis

Now we'll filter the data to focus on specific studies and polymer types with sufficient samples for analysis.

In [ ]:
# Remove wavenumber columns in the 1300-2800 cm⁻¹ range (zero-interpolated region)
# Keep only wavenumbers < 1300 and > 2800
spectral_cols = [c for c in df_norm.columns if c.replace('.', '', 1).isdigit()]
non_spectral_cols = [c for c in df_norm.columns if c not in spectral_cols]
filtered_spectral_cols = [c for c in spectral_cols if float(c) < 1300 or float(c) > 2800]
df_norm = df_norm[non_spectral_cols + filtered_spectral_cols]

print(f"\nAfter removing 1300-2800 cm⁻¹ region:")
print(f"  Removed {len(spectral_cols) - len(filtered_spectral_cols)} wavenumber columns")
print(f"  Remaining spectral columns: {len(filtered_spectral_cols)}")

print(f"\nFinal dataset shape: {df_norm.shape}")
print(f"Final sample distribution:")
print(df_norm['type'].value_counts())
print("="*80)

print("\n" + "="*80)
print("INITIALIZING FTIRdataanalysis")
print("="*80)

# Initialize the analysis class
# This class handles all visualization, statistical analysis, and machine learning
fda = FTIRdataanalysis(
    df=df_norm,
    dataset_name='Combined dataset',
    label_column="type",
    exclude_columns=['stydy', 'sample_id', 'environmental', 'resulation'],  # Exclude metadata
    random_state=42,  # For reproducibility
    n_jobs=-1,  # Use all CPU cores for parallel processing
)

print(f"\n✓ FTIRdataanalysis initialized successfully")
print(f"  Dataset: Combined dataset")
print(f"  Samples: {len(df_norm)}")
print(f"  Features: {df_norm.shape[1] - len(['type', 'study', 'sample_id', 'environmental', 'resolution'])}")
print(f"  Polymer types: {df_norm['type'].nunique()}")
print("="*80)

---

## Step 6: Dimensionality Reduction

High-dimensional spectral data (1000+ features) can be reduced to 2-3 dimensions for visualization while preserving important patterns.

### 6.1 Principal Component Analysis (PCA)

PCA finds linear combinations of features that explain maximum variance. PC1 and PC2 typically capture the most important spectral variations.

In [ ]:
# PCA: Linear dimensionality reduction
# standardize=True: Standardize features before PCA
# handle_missing="zero": Replace any NaN values with 0
# Look for:
#   - Clear separation between polymer types
#   - Explained variance in PC1 and PC2
#   - Outliers
fda.plot_pca(
    standardize=True,
    handle_missing="zero",
    figsize=(8, 8),
    save_plot=True,
    save_path=None,
)

### 6.2 t-SNE (t-Distributed Stochastic Neighbor Embedding)

t-SNE preserves local structure better than PCA, often revealing clusters not visible in PCA. Good for visualizing natural groupings.

In [ ]:
# t-SNE: Non-linear dimensionality reduction
# perplexity=50: Balance between local and global structure (5-50 typical)
# n_iter=1000: Number of optimization iterations
# pca_components=20: Pre-reduce to 20 dimensions with PCA (faster, often better)
# Look for:
#   - Tight clusters of same polymer type
#   - Separation between different types
#   - Mixed clusters may indicate similar polymers
fda.plot_tsne(
    perplexity=50,
    n_iter=1000,
    pca_components=30,
    standardize=True,
    handle_missing="zero",
    figsize=(8, 8),
    save_plot=True,
    save_path=None,
)

### 6.3 UMAP (Uniform Manifold Approximation and Projection)

UMAP is faster than t-SNE and often preserves both local and global structure better. Excellent for large datasets.

In [ ]:
# UMAP: Modern non-linear dimensionality reduction
# n_neighbors=100: Size of local neighborhood (2-200 typical)
# min_dist=0.5: Minimum distance between points in embedding (0.0-0.99)
# pca_components=20: Pre-reduce dimensionality
# Look for:
#   - Balance of local and global structure
#   - Clear polymer type clusters
#   - Overall topology of the data
fda.plot_umap(
    n_neighbors=50,
    min_dist=0.5,
    pca_components=30,
    standardize=True,
    handle_missing="zero",
    figsize=(8, 8),
    save_plot=True,
    save_path=None,
)

### 6.4 PLS-DA (Partial Least Squares Discriminant Analysis)

PLS-DA is a supervised method that finds components maximizing separation between classes. Better than PCA for classification.

In [ ]:
# PLS-DA: Supervised dimensionality reduction for classification
# n_components=20: Number of PLS components to extract
# This method uses class labels to find components that maximize separation
fda.plot_plsda(
    n_components=30,
    standardize=True,
    handle_missing="zero",
    figsize=(8, 8),
    save_plot=True,
    save_path=None,
)

### 6.5 OPLS-DA (Orthogonal PLS-DA)

OPLS-DA separates variation related to class separation from orthogonal variation (noise). Better interpretability than PLS-DA.

In [ ]:
# OPLS-DA: Orthogonal PLS-DA
# n_components=1: Predictive components
# n_orthogonal=2: Orthogonal (uncorrelated) components
# Separates class-related variation from unrelated variation
fda.plot_oplsda(
    n_components=1,
    n_orthogonal=2,
    standardize=True,
    handle_missing="zero",
    figsize=(8, 8),
    save_plot=True,
    save_path=None,
)

---

## Step 7: Clustering Analysis

Clustering discovers natural groupings in unlabeled data. Compare clustering results to true labels to assess data structure.

### 7.1 K-means Clustering

K-means partitions data into k clusters by minimizing within-cluster variance. Good for spherical, well-separated clusters.

In [ ]:
# K-means clustering
# n_clusters=8: Number of clusters (should match number of polymer types)
# pca_components=20: Reduce to 20 dimensions before clustering
# n_components_clustering=10: Use first 10 PCA components for clustering
# k_range=(2,11): Test elbow method for optimal k
# Look for:
#   - Agreement between clusters and true polymer types
#   - Elbow in within-cluster sum of squares plot
fda.plot_kmeans_clus(
    n_clusters=8,
    pca_components=30,
    n_components_clustering=50,
    k_range=(2, 11),
    standardize=True,
    handle_missing="zero",
    figsize=(8, 8),
    save_plot=True,
    save_path=None,
)

### 7.2 Hierarchical Clustering

Hierarchical clustering builds a tree of clusters (dendrogram). Useful for understanding hierarchical relationships between polymer types.

In [ ]:
# Hierarchical clustering
# n_clusters=8: Cut dendrogram to form 8 clusters
# pca_components=20: Pre-reduce dimensionality
# n_components_clustering=10: Use 10 PCA components for clustering
# n_samples_dendro=100: Plot dendrogram for 100 samples (faster)
# Look for:
#   - Dendrogram structure (which polymers cluster together)
#   - Height of merges (distance between clusters)
fda.plot_hierarchical_clus(
    n_clusters=8,
    pca_components=30,
    n_components_clustering=50,
    n_samples_dendro=100,
    standardize=True,
    handle_missing="zero",
    figsize=(8, 8),
    save_plot=True,
    save_path=None,
)

---

## Step 8: Machine Learning Classification

Now we'll train and evaluate machine learning models to classify polymer types based on FTIR spectra.

### 8.1 Prepare Data for Machine Learning

Split data into training and testing sets.

In [ ]:
print("\n" + "="*80)
print("PREPARING DATA FOR MACHINE LEARNING")
print("="*80)

# Prepare data for ML: split into training and testing sets
# test_size=0.2: Use 20% of data for testing, 80% for training
# Stratified split ensures balanced representation of each polymer type
data_dict = fda.ml_prepare_data(
    test_size=0.2,
)

print(f"\n✓ Data prepared successfully")
print(f"  Training samples: {len(data_dict['X_train'])}")
print(f"  Testing samples: {len(data_dict['X_test'])}")
print("="*80)

### 8.2 Available Models

Xpectrass provides 40 pre-configured classification models across different algorithm families.

In [ ]:
# View all available pre-configured models
# Includes: Linear models, tree-based, SVMs, neural networks, ensemble methods
print("\n" + "="*80)
print("AVAILABLE CLASSIFICATION MODELS")
print("="*80)
print(fda.available_models())
print("="*80)

### 8.3 Train and Evaluate a Single Model

Train a single model (XGBoost) and evaluate its performance.

In [ ]:
print("\n" + "="*80)
print("TRAINING SINGLE MODEL: XGBoost (100)")
print("="*80)

# Train and evaluate XGBoost model
# model_name: Use pre-configured model
# cv_folds=5: Use 5-fold cross-validation
# plot_confusion=True: Show confusion matrix
# print_test_result=True: Print detailed metrics
results = fda.run_a_model(
    model_name='XGBoost (100)',
    model=None,
    cv_folds=5,
    plot_confusion=True,
    save_plot_path='ML',
    print_test_result=True,
)

print("="*80)

### 8.4 Train and Compare All Models

Train all 35+ models and compare their performance. This identifies the best models for your data.

In [ ]:
print("\n" + "="*80)
print("TRAINING AND COMPARING ALL MODELS")
print("="*80)
print("This will train 35+ models and may take several minutes...\n")

# Train all available models and compare performance
# plot_comparision=True: Show comparison plots
# accuracy_threshold=0.9: Only show models with >90% accuracy
# top_n_methods=20: Show top 20 models
results_all = fda.run_all_models(
    plot_comparison=True,
    accuracy_threshold=0.9,
    top_n_methods=20,
    save_plot_path='ML'
)

print("\n" + "="*80)
print("MODEL COMPARISON COMPLETE")
print("="*80)

### 8.5 Hyperparameter Tuning

Fine-tune the top model(s) by optimizing hyperparameters using grid search.

In [ ]:
print("\n" + "="*80)
print("HYPERPARAMETER TUNING")
print("="*80)
print("Tuning top model(s) using grid search with cross-validation...\n")

# Perform hyperparameter tuning on the top model
# number_of_models=1: Tune only the best model
# Uses grid search with cross-validation
tuning_results = fda.model_parameter_tuning(number_of_models=1)

print("\n" + "="*80)
print("HYPERPARAMETER TUNING COMPLETE")
print("="*80)

---

## Step 9: Model Interpretation with SHAP

SHAP (SHapley Additive exPlanations) explains model predictions by showing which features (wavenumbers) contribute most to each prediction.

### 9.1 Global SHAP Analysis

Shows overall feature importance across all predictions.

In [ ]:
print("\n" + "="*80)
print("SHAP EXPLAINABILITY ANALYSIS")
print("="*80)

# Explain model predictions with SHAP
# model_name: Which model to explain
# max_display=20: Show top 20 most important features
# sample_size=100: Use 100 samples for SHAP (faster, representative)
# Generates:
#   - Summary plot (feature importance)
#   - Bar plot (mean absolute SHAP values)
#   - Beeswarm plot (feature effects)
shap_results = fda.explain_by_shap(
    model_name='XGBoost (100)',
    max_display=20,
    sample_size=100,
    test_size=0.2,
    cv_folds=5,
    save_plot_path='ML'
)

print("\n✓ SHAP analysis complete")
print("="*80)

### 9.2 Local SHAP Analysis

Explains a single prediction by showing which features pushed the model toward or away from each class.

In [ ]:
# Local SHAP plot for a single prediction
# sample_index=0: Explain prediction for first test sample
# Shows:
#   - Base value (average model output)
#   - How each feature pushes prediction toward final value
#   - Red = pushes prediction higher, Blue = pushes lower
fda.local_shap_plot(
    sample_index=0,
    figsize=(10, 8),
    save_plot_path=None
)

---

## Summary and Conclusions

### What We Accomplished

In this notebook, we performed a comprehensive analysis of FTIR spectral data:

1. ✓ **Data Loading & Filtering**: Loaded and filtered preprocessed data
2. ✓ **Exploratory Visualization**: Visualized spectral patterns and variability
3. ✓ **Statistical Analysis**: Identified discriminative wavenumbers (ANOVA, correlation)
4. ✓ **Dimensionality Reduction**: Reduced data to 2D for visualization (PCA, t-SNE, UMAP, PLS-DA, OPLS-DA)
5. ✓ **Clustering**: Discovered natural groupings (K-means, hierarchical)
6. ✓ **Machine Learning**: Trained and compared 35+ classification models
7. ✓ **Model Interpretation**: Explained predictions with SHAP

### Key Insights

From the analysis, you should now understand:

- **Spectral Signatures**: Each polymer type has characteristic FTIR peaks
- **Discriminative Features**: Which wavenumbers are most useful for classification (ANOVA, SHAP)
- **Data Structure**: How polymer types cluster in reduced dimensions
- **Model Performance**: Which models work best for your data
- **Feature Importance**: Which spectral regions drive model predictions

### Best Practices

- **Always visualize first**: Understand your data before modeling
- **Try multiple methods**: Different algorithms have different strengths
- **Use cross-validation**: Ensures robust performance estimates
- **Interpret models**: SHAP helps understand what models learn
- **Consider domain knowledge**: Combine statistical results with chemical understanding

### Next Steps

Depending on your research goals, you can:

1. **Improve Models**:
   - Try different preprocessing methods (Notebooks 1-3)
   - Perform feature selection (use ANOVA, SHAP results)
   - Try deep learning models
   - Ensemble multiple models

2. **Analyze New Samples**:
   - Use trained models to classify unknown samples
   - Calculate prediction confidence
   - Identify outliers or contaminated samples

3. **Publication**:
   - Save high-quality plots (set `save_plot=True`)
   - Export results for further analysis
   - Document your methodology

### Tips for Interpretation

- **High accuracy isn't everything**: Consider precision, recall, and F1-score
- **Confusion matrix**: Shows which polymer types are confused
- **SHAP values**: Reveal which wavenumbers matter most
- **Cross-validation scores**: Show model stability
- **Domain knowledge**: Use chemistry to validate statistical findings

### Additional Resources

- Xpectrass documentation: [docs link]
- FTIR spectroscopy guides
- Machine learning best practices
- Chemometrics literature

---

## Conclusion

You've completed a comprehensive analysis of FTIR spectral data! The visualizations, statistical analyses, and machine learning models provide deep insights into polymer classification. Use these results to inform your research and build robust classification systems.